# 05 — Calibração probabilística

## O que é um modelo calibrado?

Um modelo está **bem calibrado** se, entre todos os exemplos em que ele estima
$\hat{p}(x) = 0{,}20$, aproximadamente 20% deles são eventos reais. Formalmente:

$$P(Y=1 \mid \hat{p}(X) = p) \approx p \quad \forall\, p \in [0,1]$$

Isso importa para decisão: a regra $\hat{p}(x) > t^*$ pressupõe que $\hat{p}$
é uma probabilidade confiável, não apenas um *score* de ordenação.

Random Forests tendem a produzir probabilidades **comprimidas** — nunca muito
próximas de 0 ou 1 — porque são médias de árvores. O método **isotônico** de
calibração (Platt/isotonic regression) aprende uma transformação monótona que
mapeia os scores brutos para probabilidades bem calibradas.

## O que veremos aqui

1. Treinar um modelo base e um modelo calibrado com validação cruzada.
2. Comparar Brier Score antes e depois da calibração.
3. Visualizar a tabela de calibração: frequência empírica versus estimativa.

In [ ]:
# --- Configuração do ambiente ---
# Adicionamos src/ ao caminho de busca do Python para que os módulos
# do pacote risk_decision sejam encontrados independente de onde
# o Jupyter foi aberto.
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent  # sobe um nível se estiver dentro de notebooks/

sys.path.insert(0, str(ROOT / "src"))

# --- Importações ---
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, roc_auc_score

from risk_decision.simulation import simulate_climate_health_data
from risk_decision.calibration import calibration_table

# --- Dados ---
df = simulate_climate_health_data(n=5_000, seed=42)

X = df[["temperatura", "umidade", "vulnerabilidade"]]
y = df["evento"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

## Parte 1 — Modelo base (sem calibração)

Treinamos primeiro o Random Forest sem calibração para ter a linha de base.

In [ ]:
# --- Modelo base: Random Forest sem calibração ---
base_model = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=20,
    random_state=42,
)

base_model.fit(X_train, y_train)

# Probabilidades brutas do Random Forest
p_base = base_model.predict_proba(X_test)[:, 1]

auc_base   = roc_auc_score(y_test, p_base)
brier_base = brier_score_loss(y_test, p_base)

print("Modelo BASE (sem calibracao):")
print(f"  AUC-ROC:     {auc_base:.4f}")
print(f"  Brier Score: {brier_base:.4f}")

## Parte 2 — Modelo calibrado (isotônico)

`CalibratedClassifierCV` aplica calibração isotônica com validação cruzada:
treina o RF nos folds de treino e aprende a transformação isotônica nos folds
de validação. Isso evita que a calibração use os mesmos dados do treino,
o que causaria overfitting.

In [ ]:
# --- Modelo calibrado: RF + calibração isotônica (5-fold CV) ---
calibrated_model = CalibratedClassifierCV(
    estimator=RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=20,
        random_state=42,
    ),
    method="isotonic",   # alternativa: "sigmoid" (Platt scaling — mais suave)
    cv=5,                # 5-fold: 80% treino / 20% calibração por fold
)

calibrated_model.fit(X_train, y_train)

# Probabilidades após calibração
p_cal = calibrated_model.predict_proba(X_test)[:, 1]

auc_cal   = roc_auc_score(y_test, p_cal)
brier_cal = brier_score_loss(y_test, p_cal)

print("Modelo CALIBRADO (isotonic):")
print(f"  AUC-ROC:     {auc_cal:.4f}")
print(f"  Brier Score: {brier_cal:.4f}")
print()
print(f"Reducao no Brier Score: {brier_base - brier_cal:.4f}")
print("AUC nao muda — a calibracao nao altera a ordenacao, so a escala.")

## Parte 3 — Tabela de calibração

A tabela divide as probabilidades estimadas em 10 faixas ("bins") e mostra,
para cada faixa, qual foi a frequência empírica de eventos. Se o modelo estiver
bem calibrado, cada faixa deve ter frequência próxima ao ponto médio da faixa.


In [ ]:
# --- Tabela de calibração ---
# Compara, bin a bin, a probabilidade media estimada com a frequencia real

import pandas as pd

print("Calibracao do modelo BASE:")
tbl_base = calibration_table(y_test.values, p_base, n_bins=10)
print(tbl_base.to_string(index=False))
print()

print("Calibracao do modelo CALIBRADO:")
tbl_cal = calibration_table(y_test.values, p_cal, n_bins=10)
print(tbl_cal.to_string(index=False))
print()
print("Em um modelo perfeito, as colunas 'p_medio' e 'freq_real' seriam identicas.")

In [ ]:
# --- Gráfico de calibração: diagrama de confiabilidade ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, tbl, titulo in [
    (axes[0], tbl_base, "Modelo base (sem calibracao)"),
    (axes[1], tbl_cal,  "Modelo calibrado (isotonic)"),
]:
    p_col  = tbl.columns[0]   # coluna com probabilidade media do bin
    fr_col = tbl.columns[1]   # coluna com frequencia real do bin

    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfeito")
    ax.scatter(tbl[p_col], tbl[fr_col], s=50, zorder=5)
    ax.plot(tbl[p_col], tbl[fr_col], color="#4F81BD", linewidth=1.5)
    ax.set_xlabel("Probabilidade estimada (media do bin)")
    ax.set_ylabel("Frequencia real de eventos")
    ax.set_title(titulo)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend()

plt.tight_layout()
plt.show()
print("Quanto mais proxima a curva azul da diagonal, melhor a calibracao.")

## Resumo e o que vem a seguir

Neste notebook:

- Treinamos um modelo base e um modelo calibrado com isotonic regression.
- Vimos que a calibração reduz o Brier Score sem afetar a AUC.
- Visualizamos o diagrama de confiabilidade — a ferramenta visual de calibração.

Com um modelo bem calibrado, o limiar $t^*$ pode ser aplicado com confiança:
a probabilidade estimada pelo modelo tem o mesmo significado que a probabilidade
verdadeira que usamos nos notebooks 01–03.

**No notebook 06** vamos usar dois modelos — um com variáveis climáticas e outro
sem — e calcular o **valor da informação climática** para a decisão de saúde.

→ Próximo: `06_valor_da_informacao.ipynb`